# CS336 Assignment 2 — Colab 实验环境

这个笔记本准备 CS336 Systems 的 GPU 实验：检查 NVIDIA GPU 和 Nsight Systems、同步仓库、安装依赖，并提供可重复的 benchmark 和 profiler 函数。

在 Colab 菜单中先选 `Runtime → Change runtime type → GPU`，然后从上到下运行准备单元。实验输出默认放在 `/content`；需要跨运行时保留时，再启用 Drive 单元。

## 1. 配置 Git 仓库

`REPO_BRANCH` 指定要运行的分支。当前值是本次同步 benchmark 的分支；以后切换实验分支时只改这里。公开仓库可直接 clone。

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/BIT-Orange/cs336-assignment2-systems.git"
REPO_BRANCH = "codex/colab-benchmark"
REPO_DIR = Path("/content/cs336-assignment2-systems")


## 2. 检查 NVIDIA GPU

先确认 Colab 分配了 NVIDIA GPU，并记录型号、驱动和显存。没有 `nvidia-smi` 时，先回到 Colab 设置 GPU runtime，再重新连接。

In [ ]:
import shutil
import subprocess
import sys

print("Python:", sys.version)
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("找不到 nvidia-smi；请在 Colab 中选择 GPU runtime 并重新连接。")
gpu_info = subprocess.run(["nvidia-smi"], check=True, text=True, capture_output=True)
print(gpu_info.stdout)


## 3. 检查并安装 Nsight Systems CLI

先检查 `nsys`。若 Colab 镜像没有安装，下一单元按 NVIDIA 的 Ubuntu 软件源说明安装 CLI-only 包 `nsight-systems-cli`，不会安装桌面 GUI。来源：[NVIDIA Nsight Systems Installation Guide](https://docs.nvidia.com/nsight-systems/InstallationGuide/index.html#package-manager-installation)。

In [ ]:
import re
import shutil
import subprocess
from pathlib import Path

if shutil.which("nsys") is not None:
    print("Nsight Systems is already installed:", shutil.which("nsys"))
else:
    os_release = Path("/etc/os-release").read_text()
    version_line = next(line for line in os_release.splitlines() if line.startswith("VERSION_ID="))
    ubuntu_version = version_line.split("=", 1)[1].strip().strip('"')
    ubuntu_repo_version = ubuntu_version.replace(".", "")
    architecture = subprocess.check_output(["dpkg", "--print-architecture"], text=True).strip()

    subprocess.run(["apt-get", "update"], check=True)
    subprocess.run(
        ["apt-get", "install", "-y", "--no-install-recommends", "gnupg2", "wget", "ca-certificates"],
        check=True,
    )

    key_file = Path("/tmp/nvidia-devtools-key.pub")
    keyring_file = Path("/usr/share/keyrings/nvidia-devtools-keyring.gpg")
    key_url = "https://developer.download.nvidia.com/compute/cuda/repos/ubuntu1804/x86_64/7fa2af80.pub"
    subprocess.run(["wget", "-qO", str(key_file), key_url], check=True)
    subprocess.run(["gpg", "--dearmor", "--yes", "-o", str(keyring_file), str(key_file)], check=True)

    repo_line = (
        f"deb [signed-by={keyring_file}] "
        f"https://developer.download.nvidia.com/devtools/repos/ubuntu{ubuntu_repo_version}/{architecture}/ /"
    )
    Path("/etc/apt/sources.list.d/nvidia-devtools.list").write_text(repo_line + "\n")
    subprocess.run(["apt-get", "update"], check=True)
    subprocess.run(["apt-get", "install", "-y", "--no-install-recommends", "nsight-systems-cli"], check=True)
    key_file.unlink(missing_ok=True)
    print("Installed nsight-systems-cli for Ubuntu", ubuntu_version, architecture)


## 4. 验证 Nsight Systems

`nsys status -e` 会显示采样环境检查。即使部分 CPU 采样能力不可用，仍可以继续使用 CUDA/NVTX tracing。

In [ ]:
nsys = shutil.which("nsys")
if nsys is None:
    raise RuntimeError("Nsight Systems 安装后仍未出现在 PATH 中。")

version = subprocess.run([nsys, "--version"], check=True, text=True, capture_output=True)
print(version.stdout or version.stderr)
status = subprocess.run([nsys, "status", "-e"], check=False, text=True, capture_output=True)
print(status.stdout)
if status.stderr:
    print(status.stderr)
print("nsys status exit code:", status.returncode)


## 5. Clone 或更新仓库

新 runtime 会 clone 指定分支。重复运行时，如果仓库干净，会执行 fast-forward 更新；如果有本地改动，会保留改动并跳过 pull。

In [ ]:
import os
import subprocess

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
elif not (REPO_DIR / ".git").exists():
    raise RuntimeError(f"目标路径已存在，但不是 Git 仓库：{REPO_DIR}")
else:
    current_branch = subprocess.check_output(
        ["git", "branch", "--show-current"], cwd=REPO_DIR, text=True
    ).strip()
    dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO_DIR, text=True).strip()

    if current_branch != REPO_BRANCH:
        if dirty:
            raise RuntimeError("仓库有本地改动且分支不匹配；先保存改动，再切换 REPO_BRANCH。")
        subprocess.run(["git", "fetch", "origin", REPO_BRANCH], cwd=REPO_DIR, check=True)
        subprocess.run(
            ["git", "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"],
            cwd=REPO_DIR,
            check=True,
        )
    elif dirty:
        print("仓库有本地改动；为保留改动，本单元跳过 git pull。")
    else:
        subprocess.run(["git", "pull", "--ff-only", "origin", REPO_BRANCH], cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
print("工作目录:", Path.cwd())
print("分支:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())


## 6. 安装项目依赖

项目用 `uv` 管理依赖。首次同步可能需要下载 Python 包；同一 runtime 重复执行时会复用已安装环境。

In [ ]:
import shutil
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
uv = shutil.which("uv")
if uv is None:
    raise RuntimeError("uv 安装后仍未出现在 PATH 中。")
subprocess.run([uv, "sync"], cwd=REPO_DIR, check=True)
print("项目依赖同步完成。")


## 7. 验证 PyTorch、CUDA 和 Triton

确认 `uv` 环境中的 PyTorch 与 Colab GPU 匹配。每次更换 runtime 或依赖后都运行这一步。

In [ ]:
check_code = r'''import torch
import triton

print("PyTorch:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("Triton:", triton.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("PyTorch 没有检测到 GPU；检查 Colab runtime 与 PyTorch CUDA build。")
props = torch.cuda.get_device_properties(0)
print("GPU:", props.name)
print(f"GPU memory: {props.total_memory / (1024**3):.1f} GiB")
print("Compute capability:", f"{props.major}.{props.minor}")
'''
torch_check = subprocess.run(
    [uv, "run", "python", "-c", check_code],
    cwd=REPO_DIR,
    check=True,
    text=True,
    capture_output=True,
)
print(torch_check.stdout)


## 8. 可选：挂载 Google Drive

默认不挂载 Drive，也不会弹出授权窗口。若需要在 runtime 重置后保留结果，把 `MOUNT_DRIVE` 改成 `True` 后运行此单元。

In [ ]:
MOUNT_DRIVE = False
RESULTS_DIR = REPO_DIR / "outputs" / "colab"

if MOUNT_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    RESULTS_DIR = Path("/content/drive/MyDrive/cs336-assignment2-systems")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("实验结果目录:", RESULTS_DIR)


## 9. 重复运行 benchmark

修改下面参数，再运行代码单元中的 `run_benchmark(**BENCHMARK_RUN)`。stdout 会同时显示在单元输出并保存到带时间戳的日志文件。模型配置和参数遵循 `cs336_systems/benchmark.py`。

In [ ]:
from datetime import datetime
import shlex
import subprocess


def run_logged(command, log_path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("$", shlex.join(command))

    with log_path.open("w") as log_file:
        process = subprocess.Popen(
            command,
            cwd=REPO_DIR,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in process.stdout:
            print(line, end="")
            log_file.write(line)
        return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(f"命令失败，退出码 {return_code}。日志：{log_path}")
    print("日志:", log_path)
    return log_path


def run_benchmark(
    model_sizes=("small",),
    mode="all",
    precision="fp32",
    batch_size=4,
    context_length=512,
    warmup_steps=5,
    measurement_steps=10,
):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    sizes = list(model_sizes)
    log_path = RESULTS_DIR / "benchmarks" / f"{timestamp}_{'-'.join(sizes)}_{mode}_{precision}.log"
    command = [
        uv, "run", "python", "-m", "cs336_systems.benchmark",
        "--device", "cuda",
        "--model-sizes", *sizes,
        "--mode", mode,
        "--precision", precision,
        "--batch-size", str(batch_size),
        "--context-length", str(context_length),
        "--warmup-steps", str(warmup_steps),
        "--measurement-steps", str(measurement_steps),
    ]
    return run_logged(command, log_path)


BENCHMARK_RUN = {
    "model_sizes": ("small",),
    "mode": "all",
    "precision": "fp32",
    "batch_size": 4,
    "context_length": 512,
    "warmup_steps": 5,
    "measurement_steps": 10,
}
# Run this when ready: run_benchmark(**BENCHMARK_RUN)



def run_benchmark_sweep(
    model_sizes=("small", "medium", "large", "xl", "10b"),
    warmup_steps=(5, 0, 1, 2),
    mode="all",
    precision="fp32",
    batch_size=4,
    context_length=512,
    measurement_steps=10,
):
    """Run one model and warm-up setting at a time, saving a separate log."""
    log_paths = []

    for model_size in model_sizes:
        for warmup in warmup_steps:
            print(f"\n=== {model_size}, warmup={warmup} ===")
            try:
                log_path = run_benchmark(
                    model_sizes=(model_size,),
                    mode=mode,
                    precision=precision,
                    batch_size=batch_size,
                    context_length=context_length,
                    warmup_steps=warmup,
                    measurement_steps=measurement_steps,
                )
                log_paths.append(log_path)
            except RuntimeError as error:
                print(f"Skipping remaining warm-ups for {model_size}: {error}")
                break

    return log_paths


# Example: run_benchmark_sweep(model_sizes=("large", "xl"))


## 10. 重复运行 Nsight Systems profile

函数会用 `--capture-range=nvtx` 和 `measured_step` 标记排除 warm-up，并把 `.nsys-rep` 文件放到结果目录。默认只 profile small/full；开始时用较小模型确认 tracing 正常，再扩大范围。NVTX capture 参数见 [NVIDIA Nsight Systems User Guide](https://docs.nvidia.com/nsight-systems/UserGuide/index.html)。

In [ ]:
def run_nsys_profile(
    model_sizes=("small",),
    mode="full",
    precision="fp32",
    batch_size=4,
    context_length=512,
    warmup_steps=5,
    measurement_steps=10,
):
    nsys = shutil.which("nsys")
    if nsys is None:
        raise RuntimeError("先运行 Nsight Systems 安装与验证单元。")

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    sizes = list(model_sizes)
    output_dir = RESULTS_DIR / "nsys"
    output_dir.mkdir(parents=True, exist_ok=True)
    profile_path = output_dir / f"{timestamp}_{'-'.join(sizes)}_{mode}_{precision}"
    log_path = profile_path.with_suffix(".log")

    command = [
        nsys, "profile",
        "--trace=cuda,nvtx,osrt,cublas,cudnn",
        "--sample=none",
        "--capture-range=nvtx",
        "--nvtx-capture=measured_step",
        "--capture-range-end=none",
        "-o", str(profile_path),
        uv, "run", "python", "-m", "cs336_systems.benchmark",
        "--device", "cuda",
        "--model-sizes", *sizes,
        "--mode", mode,
        "--precision", precision,
        "--batch-size", str(batch_size),
        "--context-length", str(context_length),
        "--warmup-steps", str(warmup_steps),
        "--measurement-steps", str(measurement_steps),
        "--profile",
    ]
    run_logged(command, log_path)
    print("Nsight report:", profile_path.with_suffix(".nsys-rep"))
    return profile_path.with_suffix(".nsys-rep")


# Example: run_nsys_profile(model_sizes=("small",), mode="full")


## 11. 保存代码修改

`/content` 会在 Colab runtime 重置时清空。代码改动推送到配置的 Git 分支；benchmark 日志和 `.nsys-rep` 可复制到 Drive 保存。

In [ ]:
subprocess.run(["git", "status", "--short"], cwd=REPO_DIR, check=False)
print("修改代码后，在仓库目录中按需 git add、commit 并 push。")
